
# Updated XGBoost LOO pipeline smoke test

This notebook writes the updated `src/pipeline.py` and `src/pipeline_functions.py`, creates a small synthetic dataset, and runs a quick LOO smoke test.

Normalization options:

- `none`: leave transcriptomics unchanged.
- `zscore`: fit mean/std inside each train/eval split, including each LOO split.
- `rank`: rank-transform once outside LOO.
- `qbin5` / `qbin10`: quantile-bin once outside LOO.

The outer prediction loop is always Leave-One-Out. `tuning_strategy="nested"` is the strict version; `tuning_strategy="once"` is much faster but less strict because hyperparameters are selected once per protein.


In [18]:

# Setup paths and dependencies
import os
import sys
import subprocess
import importlib
from pathlib import Path

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
SRC_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.environ["DATA_DIR"] = str(DATA_DIR)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        pip_name = pip_name or pkg
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for pkg, pipn in [
    ("joblib", "joblib"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("sklearn", "scikit-learn"),
    ("xgboost", "xgboost"),
]:
    _ensure(pkg, pipn)

import numpy as np
import pandas as pd
from joblib import dump, load
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("DATA_DIR:", DATA_DIR)


PROJECT_ROOT: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb
SRC_DIR: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\src
DATA_DIR: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\data


In [19]:
%%writefile src/pipeline_functions.py
import itertools
import os
from typing import Optional, Sequence, Tuple, Dict, Any, List

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
import xgboost as xgb

np.random.seed(42)


# ----------------------
# Hyperparameter utilities
# ----------------------
def _all_param_combinations(param_grid: dict) -> Sequence[dict]:
    """Cartesian product of a parameter grid. Empty grid -> [{}]."""
    if not param_grid:
        return [dict()]
    keys, vals = zip(*param_grid.items())
    return [dict(zip(keys, combo)) for combo in itertools.product(*vals)]


def _sample_param_combos(param_grid: dict, max_param_combos: Optional[int]) -> Sequence[dict]:
    """Optionally sample a reproducible subset of parameter combinations."""
    combos = list(_all_param_combinations(param_grid))
    if max_param_combos is not None and len(combos) > int(max_param_combos):
        rng = np.random.default_rng(42)
        picks = rng.choice(len(combos), size=int(max_param_combos), replace=False)
        combos = [combos[i] for i in picks]
    return combos


def _default_xgb_threads(xgb_threads: Optional[int]) -> int:
    """Pick a thread count that respects SLURM when available."""
    if xgb_threads is not None and int(xgb_threads) > 0:
        return int(xgb_threads)
    slurm_threads = os.getenv("SLURM_CPUS_PER_TASK")
    if slurm_threads and slurm_threads.isdigit():
        return max(1, int(slurm_threads))
    return max(1, os.cpu_count() or 1)


def _clean_params(params: Optional[dict]) -> dict:
    """Remove metadata keys before passing parameters to XGBoost."""
    return {k: v for k, v in (params or {}).items() if not str(k).startswith("_")}


def _clone_xgb_with_params(
    params: Optional[dict],
    *,
    seed: int = 42,
    use_gpu: bool = False,
    xgb_threads: Optional[int] = None,
) -> xgb.XGBRegressor:
    """Create XGBRegressor with explicit threads and a fixed CPU/GPU backend."""
    base = dict(
        objective="reg:squarederror",
        n_jobs=_default_xgb_threads(xgb_threads),
        random_state=seed,
        verbosity=0,
        tree_method="hist",
        device="cuda" if use_gpu else "cpu",
    )
    base.update(_clean_params(params))
    return xgb.XGBRegressor(**base)


def _fit_xgb(
    params: Optional[dict],
    Xtr: np.ndarray,
    ytr: np.ndarray,
    *,
    Xva: Optional[np.ndarray] = None,
    yva: Optional[np.ndarray] = None,
    early_stopping_rounds: int = 0,
    use_gpu: bool = False,
    xgb_threads: Optional[int] = None,
) -> xgb.XGBRegressor:
    """
    Fit XGBoost while tolerating sklearn-API differences across XGBoost versions.

    Some versions accept early_stopping_rounds in fit(); newer versions prefer it
    in the constructor. This helper tries fit() first, then constructor fallback.
    """
    use_es = Xva is not None and yva is not None and early_stopping_rounds and early_stopping_rounds > 0

    if not use_es:
        est = _clone_xgb_with_params(params, use_gpu=use_gpu, xgb_threads=xgb_threads)
        est.fit(Xtr, ytr)
        return est

    est = _clone_xgb_with_params(params, use_gpu=use_gpu, xgb_threads=xgb_threads)
    try:
        est.fit(
            Xtr,
            ytr,
            eval_set=[(Xva, yva)],
            early_stopping_rounds=int(early_stopping_rounds),
            verbose=False,
        )
        return est
    except TypeError:
        params2 = dict(_clean_params(params))
        params2["early_stopping_rounds"] = int(early_stopping_rounds)
        est = _clone_xgb_with_params(params2, use_gpu=use_gpu, xgb_threads=xgb_threads)
        est.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        return est


def _best_iteration_count(est: xgb.XGBRegressor, fallback: int) -> int:
    """Return number of trees selected by early stopping, or fallback."""
    best = getattr(est, "best_iteration", None)
    if best is None:
        try:
            best = est.get_booster().best_iteration
        except Exception:
            best = None
    if best is None:
        return int(fallback)
    return max(1, int(best) + 1)  # best_iteration is zero-based.


# ----------------------
# Normalization utilities
# ----------------------
def _to_f32_series(s: pd.Series) -> np.ndarray:
    return s.to_numpy(dtype=np.float32, copy=False)


def _global_normalize_X(X: pd.DataFrame, normalization: str) -> pd.DataFrame:
    """
    Global unsupervised transformations done once before the LOO loop.

    none:
        Return X unchanged.
    zscore:
        Return X unchanged here. Z-score is fitted inside each LOO/train fold.
    rank:
        Rank-transform each gene across samples once outside LOO.
    qbin5/qbin10:
        Quantile-bin each gene once outside LOO and scale bins to [0, 1].
    """
    normalization = (normalization or "none").lower()

    if normalization in {"none", "zscore"}:
        return X

    if normalization == "rank":
        return X.rank(axis=0, method="average", pct=True).astype(np.float32)

    if normalization.startswith("qbin"):
        n_bins = int(normalization.replace("qbin", ""))
        if n_bins < 2:
            raise ValueError("qbin normalization requires at least 2 bins.")

        Xb = pd.DataFrame(index=X.index, columns=X.columns, dtype=np.float32)
        for col in X.columns:
            s = X[col]
            ranks = s.rank(method="average", pct=True)
            b = np.floor(ranks.to_numpy(dtype=np.float64) * n_bins)
            b = np.clip(b, 0, n_bins - 1)
            b = b / max(1, n_bins - 1)
            b[pd.isna(s).to_numpy()] = np.nan
            Xb[col] = b.astype(np.float32)
        return Xb

    raise ValueError(f"Unknown normalization: {normalization}")


def _zscore_fit_transform_train_test(
    X_train: pd.DataFrame,
    X_eval: pd.DataFrame,
) -> Tuple[np.ndarray, np.ndarray]:
    """Fit z-score on X_train only, then apply to X_train and X_eval."""
    Xtr = X_train.to_numpy(dtype=np.float32, copy=False)
    Xev = X_eval.to_numpy(dtype=np.float32, copy=False)

    mean = np.nanmean(Xtr, axis=0).astype(np.float32)
    std = np.nanstd(Xtr, axis=0).astype(np.float32)
    mean[~np.isfinite(mean)] = 0.0
    std[~np.isfinite(std) | (std == 0)] = 1.0

    Xtr_z = ((Xtr - mean) / std).astype(np.float32, copy=False)
    Xev_z = ((Xev - mean) / std).astype(np.float32, copy=False)
    return Xtr_z, Xev_z


def _make_train_eval_arrays(
    X_train: pd.DataFrame,
    X_eval: pd.DataFrame,
    *,
    normalization: str = "none",
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Convert train/eval data to float32 arrays.

    zscore is local: fit on X_train only.
    none/rank/qbin are already represented in the dataframe.
    """
    normalization = (normalization or "none").lower()
    if normalization == "zscore":
        return _zscore_fit_transform_train_test(X_train, X_eval)
    return (
        X_train.to_numpy(dtype=np.float32, copy=False),
        X_eval.to_numpy(dtype=np.float32, copy=False),
    )


# ----------------------
# Spearman prefilter
# ----------------------
def _spearman_abs_corr_topk(X: pd.DataFrame, y: pd.Series, k: Optional[int]) -> pd.Index:
    """Keep top-k columns by absolute Spearman correlation with y."""
    if k is None or int(k) <= 0 or X.shape[1] <= int(k):
        return X.columns
    Xr = X.rank(axis=0, method="average")
    yr = y.rank(method="average")
    s = Xr.corrwith(yr).abs().replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return s.nlargest(int(k)).index


# ----------------------
# Inner CV tuning
# ----------------------
def _fit_and_score_cv(
    param_grid: dict,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    n_splits: int,
    *,
    select_k: int = 0,
    normalization: str = "none",
    early_stopping_rounds: int = 30,
    max_param_combos: Optional[int] = None,
    use_gpu: bool = False,
    xgb_threads: Optional[int] = None,
    use_best_iteration: bool = True,
) -> dict:
    """
    KFold CV for hyperparameter tuning.

    If select_k > 0, Spearman filtering is performed inside each inner fold using
    only that fold's training samples. If X_train is already prefiltered by the
    outer split, pass select_k=0.
    """
    params_list = list(_sample_param_combos(param_grid, max_param_combos))
    if len(params_list) == 0:
        return {}

    avg_mse = np.zeros(len(params_list), dtype=float)
    best_rounds: List[List[int]] = [[] for _ in params_list]
    kf = KFold(n_splits=int(n_splits), shuffle=True, random_state=42)

    for tr_idx, va_idx in kf.split(X_train):
        Xtr_df, Xva_df = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        ytr_s, yva_s = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        if select_k and int(select_k) > 0:
            kept = _spearman_abs_corr_topk(Xtr_df, ytr_s, int(select_k))
            Xtr_df = Xtr_df[kept]
            Xva_df = Xva_df[kept]

        Xtr, Xva = _make_train_eval_arrays(Xtr_df, Xva_df, normalization=normalization)
        ytr, yva = _to_f32_series(ytr_s), _to_f32_series(yva_s)

        for k_idx, params in enumerate(params_list):
            try:
                est = _fit_xgb(
                    params,
                    Xtr,
                    ytr,
                    Xva=Xva,
                    yva=yva,
                    early_stopping_rounds=early_stopping_rounds,
                    use_gpu=use_gpu,
                    xgb_threads=xgb_threads,
                )
                pred = est.predict(Xva)
                avg_mse[k_idx] += mean_squared_error(yva, pred) / int(n_splits)
                if use_best_iteration and early_stopping_rounds and early_stopping_rounds > 0:
                    fallback = int(params.get("n_estimators", 100))
                    best_rounds[k_idx].append(_best_iteration_count(est, fallback))
            except Exception as e:
                print(f"WARNING: CV fit failed for params={params}: {type(e).__name__}: {e}")
                avg_mse[k_idx] += 1e6 / int(n_splits)

    best_idx = int(np.argmin(avg_mse))
    best_params = dict(params_list[best_idx])

    if use_best_iteration and best_rounds[best_idx]:
        original = int(best_params.get("n_estimators", 100))
        median_best = int(np.median(best_rounds[best_idx]))
        adjusted = max(25, min(original, int(np.ceil(median_best * 1.10))))
        best_params["n_estimators"] = adjusted
        best_params["_cv_median_best_iteration"] = median_best

    best_params["_cv_mse"] = float(avg_mse[best_idx])
    return best_params


# ----------------------
# Final fit and predict
# ----------------------
def _fit_final_and_predict(
    params: dict,
    X_train_sel: pd.DataFrame,
    y_train: pd.Series,
    X_test_sel: pd.DataFrame,
    *,
    normalization: str = "none",
    use_gpu: bool = False,
    xgb_threads: Optional[int] = None,
):
    Xtr, Xte = _make_train_eval_arrays(X_train_sel, X_test_sel, normalization=normalization)
    ytr = _to_f32_series(y_train)

    try:
        est = _fit_xgb(_clean_params(params), Xtr, ytr, use_gpu=use_gpu, xgb_threads=xgb_threads)
        y_pred = est.predict(Xte)

        feat_imp = None
        try:
            gain = est.get_booster().get_score(importance_type="gain")
            fmap = {f"f{i}": col for i, col in enumerate(X_train_sel.columns)}
            feat_imp = {fmap.get(k, k): float(v) for k, v in gain.items()}
        except Exception:
            pass

        return y_pred, feat_imp
    except Exception as e:
        print(f"WARNING: final fit failed: {type(e).__name__}: {e}")
        return np.array([np.nan], dtype=np.float32), None


# ----------------------
# Unified LOO pipeline
# ----------------------
def cv_pipeline_loo(
    model,  # kept for signature compatibility; not used
    param_grid: dict,
    X: pd.DataFrame,
    y: pd.DataFrame,
    protein_name: str,
    *,
    feature_selector: str = "spearman",  # kept for compatibility
    n_splits: int = 3,
    n_samples: Optional[int] = None,
    select_k: int = 500,
    preselect_k: Optional[int] = 5000,  # kept for compatibility; ignored
    early_stopping_rounds: int = 30,
    minimal_save: bool = True,
    max_param_combos: Optional[int] = None,
    use_gpu: bool = False,
    normalization: str = "none",
    tuning_strategy: str = "nested",
    xgb_threads: Optional[int] = None,
    grid_preset: str = "fast",
) -> pd.DataFrame:
    """
    Outer evaluation is always Leave-One-Out CV over samples.

    tuning_strategy='nested':
        Strict original-style LOO evaluation. Each held-out sample gets its own
        hyperparameter tuning on the outer training set.

    tuning_strategy='once':
        Faster mode. Hyperparameters are tuned once per protein and then reused
        for all LOO fits. The predictions are still LOO predictions, but the
        hyperparameter choice has seen all samples, so this is less strict for
        unbiased benchmarking.

    normalization:
        none   = raw transcriptomics.
        zscore = fitted inside each train/eval split, including each LOO split.
        rank   = rank-transformed once outside LOO.
        qbin5/qbin10 = binned once outside LOO.
    """
    if protein_name not in y.columns:
        raise ValueError(f"Protein {protein_name} not found in y columns.")

    normalization = (normalization or "none").lower()
    if normalization not in {"none", "zscore", "rank", "qbin5", "qbin10"}:
        raise ValueError(f"Invalid normalization: {normalization}")

    tuning_strategy = (tuning_strategy or "nested").lower()
    if tuning_strategy not in {"nested", "once"}:
        raise ValueError("tuning_strategy must be 'nested' or 'once'.")

    y_protein = y[protein_name]
    sample_idx = X.index if n_samples is None else X.index[:int(n_samples)]
    X = X.loc[sample_idx]
    y_protein = y_protein.loc[sample_idx]

    # Drop samples with missing target for this protein.
    valid = y_protein.notna()
    X = X.loc[valid]
    y_protein = y_protein.loc[valid]
    sample_idx = X.index

    if len(sample_idx) < max(3, int(n_splits) + 1):
        raise ValueError(
            f"Not enough non-missing samples for protein {protein_name}: "
            f"n={len(sample_idx)}, n_splits={n_splits}"
        )

    # rank/qbin are intentionally computed once outside LOO.
    # zscore is intentionally not done here; it is fitted in each train/eval split.
    X = _global_normalize_X(X, normalization)

    global_best_params = None
    if tuning_strategy == "once":
        print("Tuning hyperparameters once for this protein...")
        global_best_params = _fit_and_score_cv(
            param_grid=param_grid,
            X_train=X,
            y_train=y_protein,
            n_splits=int(n_splits),
            select_k=int(select_k),
            normalization=normalization,
            early_stopping_rounds=int(early_stopping_rounds),
            max_param_combos=max_param_combos,
            use_gpu=use_gpu,
            xgb_threads=xgb_threads,
            use_best_iteration=True,
        )
        print(f"Best once-tuned params: {global_best_params}")

    rows = []
    for i, test_id in enumerate(sample_idx, start=1):
        print(f"LOO {i}/{len(sample_idx)}: held-out sample={test_id}")

        X_train = X.drop(index=[test_id])
        y_train = y_protein.drop(index=[test_id])
        X_test = X.loc[[test_id]]
        y_test = y_protein.loc[[test_id]]

        # Spearman prefilter on the outer training set only.
        kept_features = _spearman_abs_corr_topk(X_train, y_train, int(select_k))
        Xtr_sel = X_train[kept_features]
        Xte_sel = X_test[kept_features]

        if tuning_strategy == "nested":
            # Original-style: the outer training set is prefiltered once, then
            # inner CV tunes hyperparameters only on this fixed feature set.
            best_params = _fit_and_score_cv(
                param_grid=param_grid,
                X_train=Xtr_sel,
                y_train=y_train,
                n_splits=int(n_splits),
                select_k=0,
                normalization=normalization,
                early_stopping_rounds=int(early_stopping_rounds),
                max_param_combos=max_param_combos,
                use_gpu=use_gpu,
                xgb_threads=xgb_threads,
                use_best_iteration=True,
            )
        else:
            best_params = dict(global_best_params or {})

        y_pred, feat_imp = _fit_final_and_predict(
            best_params,
            Xtr_sel,
            y_train,
            Xte_sel,
            normalization=normalization,
            use_gpu=use_gpu,
            xgb_threads=xgb_threads,
        )
        pred_val = float(y_pred[0]) if isinstance(y_pred, np.ndarray) else float(y_pred)

        if minimal_save:
            rows.append({
                "protein": protein_name,
                "LOO sample": test_id,
                "true": float(y_test.values[0]),
                "predicted": pred_val,
            })
        else:
            rows.append({
                "protein": protein_name,
                "LOO sample": test_id,
                "true": float(y_test.values[0]),
                "predicted": pred_val,
                "params_dict": {
                    **best_params,
                    "feature_selector": "spearman_prefilter",
                    "select_k": int(select_k),
                    "normalization": normalization,
                    "tuning_strategy": tuning_strategy,
                    "grid_preset": grid_preset,
                    "early_stopping_rounds": int(early_stopping_rounds),
                    "inner_cv_folds": int(n_splits),
                    "xgb_threads": _default_xgb_threads(xgb_threads),
                    "use_gpu": bool(use_gpu),
                },
                "features_coef_dict": feat_imp,
                "features_selected": list(map(str, kept_features)),
            })

    return pd.DataFrame(rows)


Overwriting src/pipeline_functions.py


In [20]:
%%writefile src/pipeline.py
#!/usr/bin/env python3
import os
import sys
import argparse
from joblib import load, dump

from pipeline_functions import cv_pipeline_loo


def parse_args():
    p = argparse.ArgumentParser(
        description="Run XGBoost LOO pipeline for a single protein."
    )

    # Required
    p.add_argument("--protein", required=True, type=str,
                   help="Protein name (column in proteomics).")

    # CV / speed controls
    p.add_argument("--splits", type=int, default=3,
                   help="Inner CV folds. Default: 3.")
    p.add_argument("--select_k", type=int, default=500,
                   help="Keep top-K features via Spearman prefilter per LOO split. 0 keeps all features.")
    p.add_argument("--early_stopping", type=int, default=30,
                   help="Early stopping rounds in inner CV. 0 disables early stopping.")
    p.add_argument("--n_samples", type=int, default=None,
                   help="If set, run only on the first N samples.")
    p.add_argument("--max_param_combos", type=int, default=None,
                   help="Randomly cap the number of hyperparameter combos explored.")
    p.add_argument("--tuning_strategy", choices=["nested", "once"], default="nested",
                   help=(
                       "nested = strict LOO: tune hyperparameters inside every LOO split; "
                       "once = faster: tune once per protein, then run LOO with fixed params. "
                       "Default: nested."
                   ))
    p.add_argument("--grid_preset", choices=["tiny", "fast", "current"], default="fast",
                   help="Hyperparameter grid size. tiny=4 combos, fast=16 combos, current=original 1944 combos.")
    p.add_argument(
        "--normalization",
        choices=["none", "zscore", "rank", "qbin5", "qbin10"],
        default="none",
        help=(
            "Transcriptomics normalization. "
            "none = raw data; "
            "zscore = fit mean/std inside each LOO train split; "
            "rank = rank-transform once outside LOO; "
            "qbin5/qbin10 = quantile-bin once outside LOO."
        ),
    )
    p.add_argument("--xgb_threads", type=int, default=None,
                   help="Threads passed to XGBoost. Default: SLURM_CPUS_PER_TASK or all visible CPUs.")

    # Convenience flags
    p.add_argument("--minimal_save", action="store_true",
                   help="Save only [protein, LOO sample, true, predicted].")
    p.add_argument("--force_gpu", action="store_true",
                   help="Use GPU with tree_method='hist', device='cuda'.")
    p.add_argument("--no_feature_selection", action="store_true",
                   help="Disable Spearman prefilter. Equivalent to --select_k 0.")
    p.add_argument("--output_suffix", default="loo_xgboost_LOO_results",
                   help="Output suffix before .joblib. Use different suffixes when comparing normalizations.")
    return p.parse_args()


def data_paths():
    """Use DATA_DIR if exported; otherwise fall back to ../data relative to this file."""
    current_dir = os.path.dirname(os.path.abspath(__file__))
    default_data_dir = os.path.abspath(os.path.join(current_dir, "..", "data"))
    data_dir = os.path.abspath(os.getenv("DATA_DIR", default_data_dir))
    X_path = os.path.join(data_dir, "transcriptomics_filtered.joblib")
    Y_path = os.path.join(data_dir, "proteomics_matched.joblib")
    return data_dir, X_path, Y_path


def build_param_grid(use_gpu: bool, preset: str = "fast"):
    """
    Backend params are fixed per run; they are not searched.

    current: original large grid, useful only for final confirmation.
    fast:    compact grid focused on the parameters that matter most.
    tiny:    very small pilot grid.
    """
    if preset == "current":
        common = {
            "max_depth": [3, 6, 8],
            "min_child_weight": [1, 3],
            "gamma": [0.0, 0.1],
            "learning_rate": [0.05, 0.1],
            "n_estimators": [600, 1200, 2000],
            "subsample": [0.7, 0.9],
            "colsample_bytree": [0.6, 0.8, 1.0],
            "reg_alpha": [0.0, 0.1, 0.5],
            "reg_lambda": [0.5, 1.0, 2.0],
            "max_bin": [256],
        }
    elif preset == "tiny":
        common = {
            "max_depth": [3, 4],
            "min_child_weight": [1, 3],
            "learning_rate": [0.05],
            "n_estimators": [600],
            "subsample": [0.9],
            "colsample_bytree": [0.8],
            "gamma": [0.0],
            "reg_alpha": [0.1],
            "reg_lambda": [1.0],
            "max_bin": [256],
        }
    else:  # fast
        common = {
            "max_depth": [3, 4],
            "min_child_weight": [1, 3],
            "learning_rate": [0.05, 0.1],
            "n_estimators": [600],
            "subsample": [0.9],
            "colsample_bytree": [0.6, 0.8],
            "gamma": [0.0],
            "reg_alpha": [0.1],
            "reg_lambda": [1.0],
            "max_bin": [256],
        }

    backend = {"tree_method": ["hist"], "device": ["cuda" if use_gpu else "cpu"]}
    return {**common, **backend}


def main():
    args = parse_args()

    if args.no_feature_selection:
        args.select_k = 0

    data_dir, X_path, Y_path = data_paths()
    print("Loading data...")
    print(f"DATA_DIR: {data_dir}")
    try:
        X = load(X_path)
        Y = load(Y_path)
        X.index.name, X.columns.name = "samples", "genes"
        Y.index.name, Y.columns.name = "samples", "proteins"
    except Exception as e:
        print(f"Error loading data ({X_path} / {Y_path}): {e}")
        sys.exit(1)

    protein = args.protein
    if protein not in Y.columns:
        print(f"Protein '{protein}' not found in proteomics.")
        sys.exit(1)

    use_gpu = bool(args.force_gpu)
    param_grid = build_param_grid(use_gpu=use_gpu, preset=args.grid_preset)

    print(
        "Running LOO XGBoost for protein "
        f"'{protein}' | grid={args.grid_preset} | tuning={args.tuning_strategy} | "
        f"normalization={args.normalization} | select_k={args.select_k} | gpu={use_gpu}"
    )

    try:
        results_df = cv_pipeline_loo(
            model=None,
            param_grid=param_grid,
            X=X,
            y=Y,
            protein_name=protein,
            n_splits=args.splits,
            n_samples=args.n_samples,
            select_k=args.select_k,
            early_stopping_rounds=args.early_stopping,
            minimal_save=args.minimal_save,
            max_param_combos=args.max_param_combos,
            use_gpu=use_gpu,
            normalization=args.normalization,
            tuning_strategy=args.tuning_strategy,
            xgb_threads=args.xgb_threads,
            grid_preset=args.grid_preset,
        )
    except Exception as e:
        print(f"Error during pipeline execution: {e}")
        sys.exit(1)

    out_path = os.path.join(data_dir, f"{protein}_{args.output_suffix}.joblib")
    try:
        print(f"Saving: {out_path}")
        try:
            dump(results_df, out_path, compress=("zstd", 3))
        except Exception:
            dump(results_df, out_path, compress=3)
        print("Done.")
    except Exception as e:
        print(f"Error saving results: {e}")
        sys.exit(1)


if __name__ == "__main__":
    main()


Overwriting src/pipeline.py


In [21]:

# Synthetic data generation
from joblib import dump
rng = np.random.default_rng(123)
n_samples, n_features, n_proteins = 18, 120, 3

X_np = rng.normal(size=(n_samples, n_features)).astype(np.float32)
w0 = np.zeros(n_features); w0[:12] = rng.normal(0, 1, size=12)
w1 = np.zeros(n_features); w1[30:45] = rng.normal(0, 1, size=15)

y0 = X_np @ w0 + rng.normal(0, 0.5, size=n_samples)
y1 = X_np @ w1 + rng.normal(0, 0.6, size=n_samples)
y2 = rng.normal(0, 1.0, size=n_samples)

samples = [f"S{i:02d}" for i in range(n_samples)]
features = [f"G{j:04d}" for j in range(n_features)]

X = pd.DataFrame(X_np, index=samples, columns=features)
Y = pd.DataFrame({"P0": y0, "P1": y1, "P2": y2}, index=samples)

# Save with the filenames expected by pipeline.py
x_path = DATA_DIR / "transcriptomics_filtered.joblib"
y_path = DATA_DIR / "proteomics_matched.joblib"
dump(X, x_path)
dump(Y, y_path)

print("Saved:", x_path)
print("Saved:", y_path)
print("Shapes:", X.shape, Y.shape)


Saved: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\data\transcriptomics_filtered.joblib
Saved: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\data\proteomics_matched.joblib
Shapes: (18, 120) (18, 3)


In [ ]:

# Direct function test: compare normalizations quickly
import importlib
import pipeline
import pipeline_functions
importlib.reload(pipeline)
importlib.reload(pipeline_functions)

from pipeline import build_param_grid
from pipeline_functions import cv_pipeline_loo

param_grid = build_param_grid(use_gpu=False, preset="tiny")
protein_name = "P0"
summary_rows = []
results_by_norm = {}

for norm in ["none", "zscore", "rank", "qbin5"]:
    print("===", norm, "===")
    res_df = cv_pipeline_loo(
        model=None,
        param_grid=param_grid,
        X=X,
        y=Y,
        protein_name=protein_name,
        n_splits=5,
        select_k=1000,
        early_stopping_rounds=10,
        max_param_combos=12,
        minimal_save=False,
        use_gpu=False,
        normalization=norm,
        tuning_strategy="nested",   # use "nested" for strict but slower benchmarking
        xgb_threads=1,
        grid_preset="tiny",
    )
    results_by_norm[norm] = res_df
    mse = mean_squared_error(res_df["true"], res_df["predicted"])
    r = pearsonr(res_df["true"], res_df["predicted"]).statistic
    summary_rows.append({"normalization": norm, "mse": mse, "pearson_r": r, "rows": len(res_df)})

summary = pd.DataFrame(summary_rows).sort_values("mse")
display(summary)


=== none ===
LOO 1/18: held-out sample=S00
LOO 2/18: held-out sample=S01
LOO 3/18: held-out sample=S02
LOO 4/18: held-out sample=S03
LOO 5/18: held-out sample=S04
LOO 6/18: held-out sample=S05
LOO 7/18: held-out sample=S06
LOO 8/18: held-out sample=S07
LOO 9/18: held-out sample=S08
LOO 10/18: held-out sample=S09
LOO 11/18: held-out sample=S10
LOO 12/18: held-out sample=S11
LOO 13/18: held-out sample=S12
LOO 14/18: held-out sample=S13
LOO 15/18: held-out sample=S14
LOO 16/18: held-out sample=S15
LOO 17/18: held-out sample=S16
LOO 18/18: held-out sample=S17
=== zscore ===
LOO 1/18: held-out sample=S00
LOO 2/18: held-out sample=S01
LOO 3/18: held-out sample=S02
LOO 4/18: held-out sample=S03
LOO 5/18: held-out sample=S04
LOO 6/18: held-out sample=S05
LOO 7/18: held-out sample=S06
LOO 8/18: held-out sample=S07
LOO 9/18: held-out sample=S08
LOO 10/18: held-out sample=S09
LOO 11/18: held-out sample=S10
LOO 12/18: held-out sample=S11
LOO 13/18: held-out sample=S12
LOO 14/18: held-out sample=S1

,normalization,mse,pearson_r,rows
3,qbin5,10.757438,0.176191,18
0,none,10.898412,0.156102,18
1,zscore,10.898412,0.156102,18
2,rank,10.898412,0.156102,18


In [23]:

# CLI smoke test using the synthetic data
import subprocess
import sys
from joblib import load

out_suffix = "smoke_none"
cmd = [
    sys.executable,
    str(SRC_DIR / "pipeline.py"),
    "--protein", "P0",
    "--splits", "2",
    "--select_k", "30",
    "--early_stopping", "5",
    "--max_param_combos", "2",
    "--grid_preset", "tiny",
    "--tuning_strategy", "once",
    "--normalization", "none",
    "--xgb_threads", "1",
    "--output_suffix", out_suffix,
    "--minimal_save",
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd, env={**os.environ, "DATA_DIR": str(DATA_DIR)})

out_path = DATA_DIR / f"P0_{out_suffix}.joblib"
cli_df = load(out_path)
display(cli_df.head())
print("Saved:", out_path)
print("Rows:", len(cli_df))


Running: c:\Users\hp\anaconda3\python.exe c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\src\pipeline.py --protein P0 --splits 2 --select_k 30 --early_stopping 5 --max_param_combos 2 --grid_preset tiny --tuning_strategy once --normalization none --xgb_threads 1 --output_suffix smoke_none --minimal_save


,protein,LOO sample,true,predicted
0,P0,S00,-0.927323,-0.668352
1,P0,S01,-4.464921,-0.720537
2,P0,S02,0.059149,-0.325051
3,P0,S03,2.636108,0.092695
4,P0,S04,1.031802,0.704840


Saved: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\pipeline_xgb\data\P0_smoke_none.joblib
Rows: 18
